# Getting started with krovlab

`roof` turns a building footprint and a pitch into a roof you can read
quantities off: faces, hips, ridges, valleys, heights, covering area.
`project` takes several cells — each its own footprint, pitch, and eave
height — and returns one takeoff. Shared walls are coincident edges: two
gables are a party wall; two pitches at the same height are one valley.
An edge can carry a knee height or a gambrel; a dormer is a small ring
on a host face.

Each example below prints the quantities, then draws:

- **plan** — brown **walls** (the building), grey **eave** (where the roof
  ends), skeleton over both
- **3D** — the solid, with the same wall line at height 0

With no overhang the two lines sit on top of each other. With an
overhang you see the roof project past the walls. `eave_height` lifts
the whole roof above datum; the brown wall line stays at height 0. A
project of two cells is one plan of both and one 3D solid of the lot.
A dormer project is not a terrain; 3D still draws.

A `Failure` has no solid; you get the input outline instead.

For plans that go wrong, see `limitations.ipynb`.


In [ ]:
from collections import defaultdict

import plotly.graph_objects as go

from krovlab import Cell, Dormer, Failure, Project, Roof, project, roof, topology_hash
from krovlab.viz import plan_view, solid_view, wavefront_steps, wavefront_view


def describe(result: Roof | Project | Failure) -> None:
    if isinstance(result, Failure):
        print(f"Failure  kind={result.kind}\n  {result.reason}")
        return
    by_kind: dict[str, list[float]] = defaultdict(list)
    for arc in result.arcs:
        by_kind[arc.kind].append(arc.length)
    print(f"terrain:           {result.validity.is_terrain}")
    print(f"ridge height:      {result.ridge_height:.3f} m")
    print(f"total sloped area: {result.total_sloped_area:.3f} m²")
    if isinstance(result, Project):
        print(f"cells:             {len(result.roofs)}")
    print(f"nodes {len(result.nodes)}, faces {len(result.faces)}, arcs {len(result.arcs)}")
    for kind, lengths in by_kind.items():
        print(f"  {kind:7s}  {len(lengths)} run(s)  total {sum(lengths):.3f} m")
    if not result.validity.is_terrain:
        for reason in result.validity.reasons:
            print(f"  !! {reason}")


def footprint_outline(
    footprint: list[tuple[float, float]],
    holes: list[list[tuple[float, float]]] | None = None,
    title: str = "",
) -> go.Figure:
    """Plan of the input rings — used when there is no roof to draw."""
    fig = go.Figure()
    if footprint:
        xs = [p[0] for p in footprint] + [footprint[0][0]]
        ys = [p[1] for p in footprint] + [footprint[0][1]]
        fig.add_trace(
            go.Scatter(
                x=xs, y=ys, mode="lines+markers", name="walls",
                line={"color": "#7c2d12", "width": 2},
                fill="toself", fillcolor="rgba(124, 45, 18, 0.10)",
                marker={"size": 7, "color": "#7c2d12"},
            )
        )
    for i, hole in enumerate(holes or []):
        if not hole:
            continue
        xs = [p[0] for p in hole] + [hole[0][0]]
        ys = [p[1] for p in hole] + [hole[0][1]]
        fig.add_trace(
            go.Scatter(
                x=xs, y=ys, mode="lines+markers",
                name=f"hole {i}" if len(holes or []) > 1 else "hole",
                line={"color": "#1d4ed8", "width": 2, "dash": "dot"},
                marker={"size": 6, "color": "#1d4ed8"},
            )
        )
    fig.update_layout(
        title=title, xaxis_title="x (m)", yaxis_title="y (m)",
        yaxis_scaleanchor="x", yaxis_scaleratio=1,
        template="plotly_white", height=420, legend_title="input",
    )
    return fig


def show(
    title: str,
    result: Roof | Project | Failure,
    footprint: list[tuple[float, float]] | None = None,
    holes: list[list[tuple[float, float]]] | None = None,
) -> None:
    """Print quantities, then draw.

    A valid roof or project: plan + 3D. Brown = walls; grey eave = where
    the roof ends. They coincide with no overhang; with an overhang the
    eaves sit outside the walls. A project draws every cell.
    A dormer project is not a terrain but 3D still draws.
    Any other constructed but non-terrain roof: plan only.
    A Failure: the input outline.
    """
    print(title)
    describe(result)
    dormer_solid = (
        isinstance(result, Project)
        and not result.validity.is_terrain
        and all("dormer" in reason.lower() for reason in result.validity.reasons)
    )
    if isinstance(result, (Roof, Project)) and (
        result.validity.is_terrain or dormer_solid
    ):
        plan = plan_view(result, walls=footprint, wall_holes=holes)
        plan.update_layout(title=f"{title} — plan (brown = walls, eave = roof edge)")
        plan.show()
        solid = solid_view(result, walls=footprint, wall_holes=holes)
        caption = (
            f"{title} — 3D (dormer: not a terrain, solid still draws)"
            if dormer_solid
            else f"{title} — 3D (brown = walls at height 0)"
        )
        solid.update_layout(title=caption)
        solid.show()
    elif isinstance(result, (Roof, Project)):
        plan = plan_view(result, walls=footprint, wall_holes=holes)
        plan.update_layout(title=f"{title} — plan (not a terrain)")
        plan.show()
    elif footprint is not None:
        footprint_outline(footprint, holes, title=f"{title} — {result.kind}").show()


## A square hip roof

A 10 m square at 45° is the smallest interesting case. The four hips
meet at an apex whose height is half the side times `tan(pitch)`:
`5 × tan(45°) = 5 m`.


In [ ]:
square = [(0.0, 0.0), (10.0, 0.0), (10.0, 10.0), (0.0, 10.0)]
square_roof = roof(square, 45.0)
assert isinstance(square_roof, Roof)
show("10 m square at 45°", square_roof, square)


### Faces

Each face rises from one footprint edge. `plan_area` is the horizontal
projection; `sloped_area` is what covering is bought by
(`plan_area / cos(pitch)`).


In [ ]:
print(f"{'edge':>4}  {'pitch':>6}  {'plan m²':>8}  {'sloped m²':>10}")
for face in square_roof.faces:
    print(
        f"{face.edge_index:4d}  {face.pitch:6.1f}  "
        f"{face.plan_area:8.3f}  {face.sloped_area:10.3f}"
    )


## A rectangle: one ridge

A 10 × 6 m rectangle at 45° has a ridge of length `10 − 6 = 4 m` at
height 3 m.


In [ ]:
rectangle = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)]
rect_roof = roof(rectangle, 45.0)
assert isinstance(rect_roof, Roof)
show("10 × 6 m rectangle at 45°", rect_roof, rectangle)


## Pitch spellings

`100%` is `1:1` is `45°`. US `4:12` is `atan(4/12) ≈ 18.4°`.


In [ ]:
for spelling in (45.0, (1, 1), "1:1", "100%", "4:12"):
    result = roof(square, spelling)
    assert isinstance(result, Roof)
    print(f"{spelling!r:>8}  ridge {result.ridge_height:7.3f} m  pitch {result.faces[0].pitch:.3f}°")

shallow = roof(square, "4:12")
assert isinstance(shallow, Roof)
show("10 m square at 4:12", shallow, square)


## A different pitch per wall


In [ ]:
mixed = roof(square, [60.0, 45.0, 60.0, 45.0])
assert isinstance(mixed, Roof)
show("square, pitches [60, 45, 60, 45]", mixed, square)


## L and U: valleys


In [ ]:
l_shape = [
    (0.0, 0.0), (10.0, 0.0), (10.0, 6.0),
    (3.0, 6.0), (3.0, 10.0), (0.0, 10.0),
]
u_shape = [
    (0.0, 0.0), (10.0, 0.0), (10.0, 8.0), (7.0, 8.0),
    (7.0, 3.0), (3.0, 3.0), (3.0, 8.0), (0.0, 8.0),
]
show("L-shape at 45°", roof(l_shape, 45.0), l_shape)
show("U-shape at 45°", roof(u_shape, 45.0), u_shape)


## A courtyard

A hole is a second ring at **eave height**. The roof does not cover it.
A single pitch covers the hole edges too — you do not pass a separate
pitch for the courtyard unless you want one.

A 10 m square with a centred 4 m courtyard at 45°: ridge 1.5 m, plan
area 84 m².


In [ ]:
courtyard = [(3.0, 3.0), (7.0, 3.0), (7.0, 7.0), (3.0, 7.0)]
court_roof = roof(square, 45.0, holes=[courtyard])
assert isinstance(court_roof, Roof)
show("square with 4 m courtyard at 45°", court_roof, square, [courtyard])
print("faces (outer 0–3, hole 4–7):", [(f.edge_index, f.pitch) for f in court_roof.faces])


## Can a hole be a circle?

Not as a true curve — every wall is a straight segment. Approximate a
circle with an n-gon. You get **one inward face per segment**, not a
cone of revolution. More sides look rounder and cost more faces.


In [ ]:
import math

def circle(cx: float, cy: float, radius: float, n: int = 16) -> list[tuple[float, float]]:
    return [
        (cx + radius * math.cos(2 * math.pi * i / n),
         cy + radius * math.sin(2 * math.pi * i / n))
        for i in range(n)
    ]

round_well = circle(5.0, 5.0, 1.5, n=16)
round_roof = roof(square, 45.0, holes=[round_well])
assert isinstance(round_roof, Roof)
show("16-sided circular courtyard, r = 1.5 m", round_roof, square, [round_well])
print(f"{len(round_roof.faces)} faces: 4 outer + 16 on the hole")


## A hole without its own pitch — and why that is not a chimney

Two different requests get mixed up here.

**"I don't want to specify a pitch for the hole."** A scalar pitch
already applies to every edge, including the hole. The courtyard above
used `roof(square, 45, holes=[...])` — no list.

**"A cut-through, like a chimney."** A chimney is a hole *through the
slope*, above the eaves. This library's hole is a courtyard at eave
height: the roof still grows inward faces down to that opening. A
chimney, dormer or rooflight is a penetration, and is not in the model.

The closest thing you *can* do is gable every hole edge (`pitch = 90`):
vertical inner walls, no inward-sloping faces. That is a light well
with vertical sides, still opening at eave height, still not a chimney.


In [ ]:
# Outer 45°, hole 90° — vertical courtyard walls.
well_pitches = [45.0, 45.0, 45.0, 45.0, 90.0, 90.0, 90.0, 90.0]
well = roof(square, well_pitches, holes=[courtyard])
assert isinstance(well, Roof)
show("courtyard with vertical inner walls (not a chimney)", well, square, [courtyard])
print("faces only on outer edges:", [f.edge_index for f in well.faces])


## Gable ends and a shed

`pitch = 90` on an outer edge is a gable. Three gables leave a shed.
Gabling every outer edge cannot close.


In [ ]:
show("rectangle, east wall gabled", roof(rectangle, [45.0, 90.0, 45.0, 45.0]), rectangle)
show("rectangle, both short sides gabled", roof(rectangle, [45.0, 90.0, 45.0, 90.0]), rectangle)
show("rectangle shed (three gables)", roof(rectangle, [45.0, 90.0, 90.0, 90.0]), rectangle)
show("every edge a gable", roof(square, 90.0), square)


## Eaves overhang — walls vs roof edge

`overhang` is metres **past the walls**. The library offsets the
footprint outward (and each hole inward) and roofs that larger polygon.

In the drawings:

- **brown `walls`** — the building you passed in
- **grey `eave`** — where the roof ends
- the band between them is the overhang

On a 10 × 6 m rectangle with 0.5 m overhang the eaves run from
(−0.5, −0.5) to (10.5, 6.5). A courtyard hole shrinks: a 4 m well
becomes 3 m. An overhang that closes a hole or folds a thin wing is a
named `Failure`.


In [ ]:
overhung = roof(rectangle, 45.0, overhang=0.5)
assert isinstance(overhung, Roof)
show("rectangle, 0.5 m overhang — eaves outside the walls", overhung, rectangle)

eaves = [a for a in overhung.arcs if a.kind == "eave"]
print("eave corners (roof edge):")
for arc in eaves:
    n = overhung.nodes[arc.start]
    print(f"  ({n.x:.2f}, {n.y:.2f})")
print("walls stay at x = 0..10, y = 0..6")


In [ ]:
overhung_court = roof(square, 45.0, holes=[courtyard], overhang=0.5)
assert isinstance(overhung_court, Roof)
show("courtyard, 0.5 m overhang — outer eaves out, hole eaves in", overhung_court, square, [courtyard])

show(
    "overhang that closes the 4 m courtyard",
    roof(square, 45.0, holes=[courtyard], overhang=3.0),
    square,
    [courtyard],
)


## Eave height — a plate above datum

`eave_height` is metres **above datum**. The roof is built and checked
at the eave plane, then every node is lifted by that constant. Default
is zero: omitting it is the same as `eave_height=0`.

On the 10 × 6 m rectangle at 45°, the ridge is 3 m above the eaves.
`eave_height=7` puts that ridge at 10 m. Plan areas do not change. In
the 3D view the brown wall line stays at height 0, so you see the
plate.

In [ ]:
lifted = roof(rectangle, 45.0, eave_height=7.0)
assert isinstance(lifted, Roof)
show("rectangle on a 7 m plate — ridge at 10 m", lifted, rectangle)
print(f"lowest node (eaves): {min(n.height for n in lifted.nodes):.1f} m")
print(f"ridge height:        {lifted.ridge_height:.1f} m  (3 m at datum + 7 m plate)")
print(
    f"plan area unchanged: {sum(f.plan_area for f in lifted.faces):.1f} m²"
)

## A project of two cells

Two detached 5 × 6 m hips, eave heights 5 m and 7 m, pitch 45°. Each
ridge is 2.5 m above its eave, so the project ridge height is 9.5 m.
Plan areas sum to 60 m². This is two wings, not one skeleton over the
outer wall — an L drawn as one polygon would still be one cell.

Overlapping cells are a named Failure, not a double-counted takeoff.


In [ ]:
house = [(0.0, 0.0), (5.0, 0.0), (5.0, 6.0), (0.0, 6.0)]
garage = [(8.0, 0.0), (13.0, 0.0), (13.0, 6.0), (8.0, 6.0)]
built = project(
    [
        Cell(house, 45.0, eave_height=5.0),
        Cell(garage, 45.0, eave_height=7.0),
    ]
)
assert isinstance(built, Project)
show("two detached 5 × 6 m hips at 5 m and 7 m plates", built)
print(f"plan area: {sum(face.plan_area for face in built.faces):.1f} m²")
for face in built.faces:
    print(f"  cell {face.cell_index}  edge {face.edge_index}  plan {face.plan_area:.1f} m²")

show(
    "overlapping cells",
    project([Cell(house, 45.0), Cell([(2.0, 0.0), (7.0, 0.0), (7.0, 6.0), (2.0, 6.0)], 45.0)]),
    house,
)


## Concatenated gables and a valley

Shared walls are coincident edges, not a join argument. Two 5 × 6 m
gable cells sharing the party wall at x = 5, plate heights 5 m and 7 m:
each ridge is 3 m above its eave (half the 6 m gable-wall span), so the
project ridge is 10 m. The party wall is not counted twice as eaves.

Two hips facing a common inner eave at the same height meet as one
valley — an M-roof. A gable against a pitch, or pitched eaves at two
heights, is a named Failure.

The web demo (`uv run --extra web python -m web`) has the same pair as
**Two gables sharing a wall**. Pick that example, or add a cell on a
selected wall. **Update roof** is still one form POST.


In [ ]:
gables = [45.0, 90.0, 45.0, 90.0]
low = [(0.0, 0.0), (5.0, 0.0), (5.0, 6.0), (0.0, 6.0)]
high = [(5.0, 0.0), (10.0, 0.0), (10.0, 6.0), (5.0, 6.0)]
pair = project(
    [
        Cell(low, gables, eave_height=5.0),
        Cell(high, gables, eave_height=7.0),
    ]
)
assert isinstance(pair, Project)
show("two concatenated gables at 5 m and 7 m plates", pair)
eave_m = sum(arc.length for arc in pair.arcs if arc.kind == "eave")
print(f"eaves: {eave_m:.1f} m (party wall not counted twice)")

m_roof = project([Cell(low, 45.0), Cell(high, 45.0)])
assert isinstance(m_roof, Project)
show("two hips sharing an inner eave: one valley", m_roof)
print(
    "valley:",
    sum(arc.length for arc in m_roof.arcs if arc.kind == "valley"),
    "m",
)

show(
    "gable versus pitch on a shared wall",
    project([Cell(low, gables), Cell(high, 45.0)]),
    low,
)
show(
    "pitched shared eave at two plate heights",
    project(
        [
            Cell(low, 45.0, eave_height=5.0),
            Cell(high, 45.0, eave_height=7.0),
        ]
    ),
    low,
)


## Knee height: a gablet, a Dutch, a half-hip

`knee_height` is metres of **vertical wall** on an edge before that
edge's pitch begins. Half-hip, Dutch gable, and gablet are this one
knob at different heights, plus a gable or a remaining hip as the
pitch on that edge. Zero is the same roof as omitting it.

The 10 × 6 m rectangle at 45° has a 3 m ridge. Knee 3 m on the east
short edge is a vertical gablet: no face there, neighbours close as
verges, ridge height still 3 m. On the web demo, pick **Knee (gablet)**
or set that wall's type to Knee.

A gable (`pitch = 90`) plus a knee on the same edge is
`gable_versus_knee`. A knee on one cell of a two-cell project leaves
the other cell unchanged.


In [ ]:
un_kneed = roof(rectangle, 45.0)
gablet = roof(rectangle, 45.0, knee_height=[0.0, 3.0, 0.0, 0.0])
assert isinstance(un_kneed, Roof)
assert isinstance(gablet, Roof)
show("rectangle, 3 m knee on the east short wall", gablet, rectangle)
print(f"ridge height: {gablet.ridge_height:.1f} m  (same as un-kneed {un_kneed.ridge_height:.1f} m)")
print(f"faces: {[face.edge_index for face in gablet.faces]}  (no face on the kneed edge)")
print(f"verges: {sum(1 for arc in gablet.arcs if arc.kind == 'verge')}")
print(
    "ridge length: "
    f"{next(arc.length for arc in gablet.arcs if arc.kind == 'ridge'):.1f} m"
)

show(
    "gable plus knee on the same edge",
    roof(rectangle, [45.0, 90.0, 45.0, 45.0], knee_height=[0.0, 3.0, 0.0, 0.0]),
    rectangle,
)


## Gambrel: steep then shallow up the wall

A gambrel is three numbers on an edge: steep pitch, shallow pitch, and
break height in metres **above that cell's eave**. The takeoff reports
two faces on that wall. A barn break is those three numbers.

The long walls of the 10 × 6 m rectangle below are 60° then 30°, with
the break at √3 m so the steep band insets 1 m. Short walls stay 45°.
Plan areas still sum to the footprint.

One story per edge: gambrel plus knee, or gambrel plus gable, is a
named Failure. Pick **Gambrel (barn)** on the web demo, or set a wall
to Gambrel.


In [ ]:
break_height = 3**0.5
barn = roof(
    rectangle,
    45.0,
    gambrel=[
        (60.0, 30.0, break_height),
        None,
        (60.0, 30.0, break_height),
        None,
    ],
)
assert isinstance(barn, Roof)
show("rectangle, long walls 60° then 30° at √3 m", barn, rectangle)
print(f"faces: {len(barn.faces)}  (two per long wall, one per short)")
for edge in (0, 2):
    pair = [face for face in barn.faces if face.edge_index == edge]
    print(
        f"  edge {edge}: {pair[0].pitch:.0f}° plan {pair[0].plan_area:.3f} m², "
        f"{pair[1].pitch:.0f}° plan {pair[1].plan_area:.3f} m²"
    )
print(f"plan areas sum: {sum(face.plan_area for face in barn.faces):.1f} m²")

show(
    "gambrel plus knee on the same edge",
    roof(
        rectangle,
        45.0,
        knee_height=[1.0, 0.0, 0.0, 0.0],
        gambrel=[(60.0, 30.0, break_height), None, None, None],
    ),
    rectangle,
)


## A dormer on a host face

A dormer is extra input to `project`: which cell, a plan ring that sits
on exactly one host face, and a pitch list. The child is `roof` on that
ring, lifted onto the host plane — its eave is the intersection with
the host, not the building eave. Host sloped area loses the opening;
dormer faces add.

A gable dormer and a shed dormer are the same placement with different
pitches. A project with dormers is **not** a terrain (the covering has
a hole); quantities still add up and **3D still draws**. That is the
documented exception to hiding 3D on a non-terrain. A plus-shape still
hides 3D — see `limitations.ipynb`.

A ring that overlaps two faces, or that lies outside the host, is a
named Failure. On the web demo, pick **Dormer on a hip**; the dormer
rectangle is already there to drag.


In [ ]:
dormer_ring = [(4.0, 0.5), (6.0, 0.5), (6.0, 2.0), (4.0, 2.0)]
gable_dormer = [45.0, 90.0, 45.0, 90.0]
shed_dormer = [45.0, 90.0, 90.0, 90.0]
host = project([Cell(rectangle, 45.0)])
built = project(
    [Cell(rectangle, 45.0)],
    [Dormer(0, dormer_ring, gable_dormer)],
)
assert isinstance(host, Project)
assert isinstance(built, Project)
show("10 × 6 m hip plus a 2 × 1.5 m gable dormer", built, rectangle)
south = next(face for face in host.faces if face.edge_index == 0)
south_after = next(face for face in built.faces if face.edge_index == 0)
print(
    f"south sloped: {south.sloped_area:.3f} → {south_after.sloped_area:.3f} m² "
    f"(opening {south.sloped_area - south_after.sloped_area:.3f} m²)"
)
print(f"faces: {len(host.faces)} host → {len(built.faces)} with dormer")
print(f"terrain: {built.validity.is_terrain}  ({built.validity.reasons[0]})")

shed = project([Cell(rectangle, 45.0)], [Dormer(0, dormer_ring, shed_dormer)])
assert isinstance(shed, Project)
show("same ring, shed pitches", shed, rectangle)

show(
    "dormer overlapping two faces",
    project(
        [Cell(rectangle, 45.0)],
        [Dormer(0, [(4.5, 2.0), (5.5, 2.0), (5.5, 4.0), (4.5, 4.0)], gable_dormer)],
    ),
    rectangle,
)


## When the input cannot be roofed

`Failure` is a value. Each refusal is drawn as the input outline.


In [ ]:
cases = (
    ("out of range", square, 0.0, {}),
    ("bowtie", [(0.0, 0.0), (10.0, 10.0), (10.0, 0.0), (0.0, 10.0)], 45.0, {}),
    ("a line", [(0.0, 0.0), (10.0, 0.0), (4.0, 0.0)], 45.0, {}),
    (
        "parallel edges, two pitches",
        [(0.0, 0.0), (5.0, 0.0), (10.0, 0.0), (10.0, 6.0), (0.0, 6.0)],
        [45.0, 30.0, 45.0, 45.0, 45.0],
        {},
    ),
    (
        "hole touching the outer ring",
        square, 45.0,
        {"holes": [[(0.0, 0.0), (4.0, 0.0), (4.0, 4.0), (0.0, 4.0)]]},
    ),
)
for label, footprint, pitch, kwargs in cases:
    show(label, roof(footprint, pitch, **kwargs), footprint, kwargs.get("holes"))


## Validity, events, wavefront


In [ ]:
print(f"is_terrain: {rect_roof.validity.is_terrain}")
print(f"topology hash: {topology_hash(rect_roof)}")

logged = roof(l_shape, 45.0, events=True)
assert not isinstance(logged, Failure)
built, events = logged
print(f"{len(events)} events on the L-shape")
wavefront_view(rect_roof, 1.5, walls=rectangle).update_layout(
    title="rectangle wavefront at 1.5 m"
).show()
for fig in wavefront_steps(built, events, walls=l_shape):
    fig.show()


## What is here, and what is not

Convex, L, U, polygonal (including circular-ish) courtyards, per-edge
pitch, gables, overhang with walls drawn inside the eaves, eave height
on a single roof, a project of several cells, concatenated gables and
a valley, knee height (gablet / half-hip / Dutch), gambrel, a dormer
on a host face, and those same examples on the web demo.

Not a chimney or any other hole *through the slope* that is not a
dormer. Not a true circular wall — only an n-gon. See
`limitations.ipynb`.
